# Enveda CASMI 2026 — Spectral-Library Cosine-Search Baseline (Precursor-Mass Blocked)

**Task**: predict the 2D structure (SMILES) of unknown molecules from MS/MS spectra, submit up to 25 ranked
candidates per `molecule_id`, scored with **MRR@25** against InChIKey14 (tautomer-canonicalized) match.

**Approach (this notebook)** — a classic spectral-library search baseline:

1. Build a reference spectral library from `train.parquet`, restricted to the libraries whose **chemistry**
   resembles the test set — natural products / bioactive natural-product-adjacent compounds — rather than
   libraries that merely share the same instrument. `enveda-180` is the same Bruker timsTOF as the test set,
   but it is a synthetic drug-like screening collection: an earlier version of this notebook used it as the
   primary reference and scored **Public MRR@25 = 0.000**, because none of its ~183k drug-like structures are
   the natural-product answers the test set is asking for, and its sheer size (94% of the combined library)
   drowned out the small natural-product-relevant libraries in both the ranked candidates *and* the fallback
   list. See the reference notes' library table (§5) for why `gnps`, `riken`, `pluskal_ms2`, etc. are the
   better fit despite being acquired on different instruments — chemical relevance matters far more than
   instrument match for this task.
2. For every test spectrum, convert `precursor_mz` + `adduct` into a **neutral monoisotopic mass** and use it
   to *block* the search — only reference spectra within a small ppm/Da window are ever compared. This turns an
   otherwise O(test × train) similarity search into something that runs in minutes on CPU.
3. Within each mass window, score candidates with a **greedy-matched cosine similarity** on the raw peak lists.
4. Aggregate scores across all spectra belonging to a `molecule_id` (multiple collision energies / adducts),
   dedupe candidate structures by InChIKey14, and keep the best 25.
5. Pad with a global "most common reference structure" fallback so every molecule always gets at least one guess.

**Expected behaviour w.r.t. novelty classes** (hidden during the competition, see reference notes §1):
this method is strong on **Novelty Class 1** (structure has public reference spectra) because it is literally
doing spectral-library matching. It degrades on **Class 2** (known structure, no public spectra — no spectral
match exists, only the fallback list helps) and is essentially blind to **Class 3** (structure not even in
PubChem — no amount of spectral matching can invent a genuinely novel scaffold). Good next steps to extend
this baseline: broaden the reference library further, swap the naive cosine for a *modified* cosine that also
matches neutral-loss-shifted peaks, or move to a formula-first / de-novo model (MIST-CF, SIRIUS, or a
DreaMS-style pretrained spectrum embedding) for Class 2/3 molecules.

**Environment notes**
- No `pip install` / network calls anywhere in this notebook — everything below uses packages that ship with
  the standard Kaggle Python image (`pandas`, `numpy`, `pyarrow`, `rdkit`, `matplotlib`, `tqdm`). This keeps the
  notebook safe to run with **Internet: Off**, which is required at submission time. (RDKit is optional — see
  the import cell below; if it's missing, only the local MRR@25 diagnostic section is skipped.)
- This pipeline is pure CPU/pandas/numpy — no GPU is used, so a CPU-only Kaggle session is enough.
- Dataset paths follow the competition's mounted location exactly:
  `/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/`.


In [ ]:
# Core imports. Everything here ships with the default Kaggle Python image — no pip installs needed,
# so this notebook stays compatible with "Internet: Off" sessions (required at submission time).
import os
import random
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
# RDKit is ONLY needed for the local MRR@25 validation section further down (it mirrors the official
# scoring: tautomer canonicalization -> InChIKey14). The core spectral-search pipeline and submission.csv
# generation do NOT depend on RDKit at all, so a missing RDKit install never blocks producing a submission.
#
# Some Kaggle Docker image variants ship RDKit out of the box, others don't. This cell:
#   1. tries a plain `import rdkit` first (works whenever it's preinstalled — the common case),
#   2. if that fails AND internet is reachable, tries a one-off `pip install rdkit` (fine for interactive
#      / Internet:On development, but this branch is skipped automatically at Internet:Off submission time
#      since the install itself will simply fail fast and we catch that),
#   3. otherwise sets RDKIT_AVAILABLE = False and the notebook keeps going — only the local-validation
#      section will be skipped, everything needed for submission.csv still runs.
#
# If you want RDKit available every time (recommended so the local MRR@25 proxy actually runs): in the
# Notebook's right-hand "Settings" panel, check Environment -> "Always use latest environment", which on
# most Kaggle images ships RDKit preinstalled. Alternatively, attach a Kaggle Dataset with an offline RDKit
# wheel and swap the pip command below for `pip install --no-index --find-links=/kaggle/input/<dataset> rdkit`.
RDKIT_AVAILABLE = True
try:
    import rdkit
    from rdkit import Chem
    from rdkit.Chem.MolStandardize import rdMolStandardize
except ModuleNotFoundError:
    print("rdkit not found in this environment. Attempting a one-off pip install "
          "(only works if Internet is ON for this session)...")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True, timeout=180)
        import rdkit
        from rdkit import Chem
        from rdkit.Chem.MolStandardize import rdMolStandardize
        print("rdkit installed successfully.")
    except Exception as install_err:
        RDKIT_AVAILABLE = False
        print(f"Could not install rdkit ({install_err}).")
        print("The core library-search pipeline and submission.csv generation do NOT require rdkit and "
              "will still run below. The 'local MRR@25 validation' section will be skipped since it needs "
              "RDKit's tautomer canonicalization to mirror the official scoring. See the comment above this "
              "cell for two ways to get RDKit available in this environment.")

# The competition's evaluation pins RDKit 2026.03.3 for tautomer canonicalization / InChIKey14 matching.
# We only WARN if the installed version differs — the pipeline still runs, but the local MRR@25 proxy
# computed later in this notebook may differ slightly from the leaderboard score if canonicalization
# behavior changed between RDKit versions.
PINNED_RDKIT_VERSION = "2026.03.3"
if RDKIT_AVAILABLE:
    print(f"Installed RDKit version: {rdkit.__version__}")
    if rdkit.__version__ != PINNED_RDKIT_VERSION:
        print(f"  -> WARNING: differs from the pinned scoring version ({PINNED_RDKIT_VERSION}). "
              f"Local MRR@25 below is an approximation of the official score.")
else:
    print("RDKit unavailable -> local MRR@25 validation section will be skipped.")


In [ ]:
# Official competition paths (exact, as provided).
COMPETITION_DIR = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
TRAIN_PATH = f"{COMPETITION_DIR}/train.parquet"
TEST_PATH = f"{COMPETITION_DIR}/test.parquet"
SAMPLE_SUB_PATH = f"{COMPETITION_DIR}/sample_submission.csv"

for p in [COMPETITION_DIR, TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    print(p, "->", "OK" if os.path.exists(p) else "MISSING")


## 1. Load the test set and take a first look

We load `test.parquet` in full (it's small — ~1,500 spectra) and `sample_submission.csv` to confirm the
expected molecule count and formatting before touching the much larger `train.parquet`.


In [ ]:
test_df = pd.read_parquet(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("test.parquet shape:", test_df.shape)
print("unique molecule_id in test:", test_df["molecule_id"].nunique())
print("unique molecule_id in sample_submission:", sample_sub["molecule_id"].nunique())
test_df.head(3)


In [ ]:
# Sanity-check the schema against the reference spec (12 documented test columns).
expected_test_cols = [
    "molecule_id", "spectrum_id", "ms2_mzs", "ms2_normalized_intensities", "base_peak_intensity",
    "adduct", "ionization_mode", "instrument_type", "precursor_mz",
    "collision_energy_orig", "collision_energy_ev", "collision_energy_orig_units",
]
missing = set(expected_test_cols) - set(test_df.columns)
extra = set(test_df.columns) - set(expected_test_cols)
print("Missing expected columns:", missing or "none")
print("Unexpected extra columns:", extra or "none")
print("instrument_type unique values in test (spec says always 'timsTOF'):", test_df["instrument_type"].unique())


## 2. EDA — spectra counts, molecular weight, adducts

A quick look at the shape of the test set before building the reference library, mainly to sanity-check the
values from the reference notes and to spot anything that would change the blocking/tolerance strategy below.


In [ ]:
# --- Spectra-per-molecule distribution (spec says 1-16 spectra/molecule, median 3) ---
spectra_per_molecule = test_df.groupby("molecule_id").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(spectra_per_molecule, bins=range(1, 18), edgecolor="black", align="left")
axes[0].set_title("Spectra per molecule_id (test)")
axes[0].set_xlabel("# spectra")
axes[0].set_ylabel("# molecules")

axes[1].bar(["min", "median", "mean", "max"],
            [spectra_per_molecule.min(), spectra_per_molecule.median(),
             spectra_per_molecule.mean(), spectra_per_molecule.max()])
axes[1].set_title("Spectra-per-molecule summary stats")
plt.tight_layout()
plt.show()

print(spectra_per_molecule.describe())


In [ ]:
# --- Adduct and ionization-mode distribution (test) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

test_df["adduct"].value_counts().plot(kind="barh", ax=axes[0])
axes[0].set_title("Adduct distribution (test spectra)")
axes[0].invert_yaxis()

test_df["ionization_mode"].value_counts().plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Ionization mode (test spectra)")

plt.tight_layout()
plt.show()


## 3. Neutral monoisotopic mass from `precursor_mz` + `adduct`

Test spectra can be measured under 7 different adducts, and reference-library spectra may be measured under a
different adduct still. To compare spectra of the *same underlying molecule* regardless of how it was ionized,
everything is converted to a common currency: **neutral monoisotopic mass**. This is the key that mass-window
blocking (used both for EDA below and for the actual search) is built on.

Electron mass (~0.00055 Da) is neglected throughout — negligible for windowed blocking at the ppm tolerances
used here, but this is *not* meant to be an exact-mass calculator.


In [ ]:
PROTON = 1.007276
ELECTRON = 0.000549
NA = 22.989770
K = 38.963158
CL = 34.968853
NH4 = 18.033823
H2O = 18.010565
FORMIC_ACID = 46.005480

# mass shift = ion m/z - neutral monoisotopic mass, for every adduct documented in the reference notes.
ADDUCT_MASS_SHIFT = {
    "[M+H]+": PROTON,
    "[M+NH4]+": NH4,
    "[M-H2O+H]+": PROTON - H2O,
    "[M-2H2O+H]+": PROTON - 2 * H2O,
    "[M+Na]+": NA - ELECTRON,
    "[M+K]+": K - ELECTRON,
    "[M-H]-": -PROTON,
    "[M-H2O-H]-": -H2O - PROTON,
    "[M+CH2O2-H]-": FORMIC_ACID - PROTON,
    "[M+Cl]-": CL + ELECTRON,
}

def neutral_mass_from_precursor(precursor_mz, adduct):
    # Convert a measured precursor m/z + adduct string into a neutral monoisotopic mass estimate.
    shift = ADDUCT_MASS_SHIFT.get(adduct)
    if shift is None or precursor_mz is None or (isinstance(precursor_mz, float) and np.isnan(precursor_mz)):
        return np.nan
    return precursor_mz - shift

unmapped = set(test_df["adduct"].unique()) - set(ADDUCT_MASS_SHIFT)
print("Adducts in test not covered by ADDUCT_MASS_SHIFT:", unmapped or "none")

test_df["neutral_mass"] = [
    neutral_mass_from_precursor(mz, add) for mz, add in zip(test_df["precursor_mz"], test_df["adduct"])
]


In [ ]:
# --- Molecular weight (neutral mass) distribution, one point per test spectrum ---
plt.figure(figsize=(8, 4))
plt.hist(test_df["neutral_mass"].dropna(), bins=40, edgecolor="black")
plt.title("Neutral monoisotopic mass distribution (test spectra)")
plt.xlabel("Neutral mass (Da)")
plt.ylabel("# spectra")
plt.show()

print(test_df["neutral_mass"].describe())
print("Reference spec says the monoisotopic mass range is 157-1159 Da (median 348) -- compare against the above.")


## 4. Build the reference spectral library from `train.parquet`

`train.parquet` is 3 GB / ~2M spectra, so we never load it in full for the search pipeline. Two things keep
this tractable:

- We only read the columns we actually need (`pyarrow`'s columnar format makes this cheap).
- For the **library search itself** we restrict `REFERENCE_LIBS` to libraries whose chemistry resembles the
  test set's natural-product focus (see reference notes §1 "Focus"), rather than picking by instrument match
  alone.

> **Why not `enveda-180`?** It's tempting to pick it because it's acquired on the exact same Bruker timsTOF as
> the test set — but per the reference notes it is a **synthetic drug-like screening library**, chemically
> unrelated to natural products. An earlier version of this notebook used `["enveda-np-examples", "enveda-180"]`
> as the reference and scored **Public MRR@25 = 0.000**: `enveda-180` supplies 94% of that combined library
> (1.15M of 1.23M spectra), so it dominated both the ranked candidates *and* the "most common structure"
> fallback with confidently-wrong drug-like SMILES, while contributing essentially zero correct natural-product
> answers. Instrument match matters for spectral *fragmentation pattern* similarity, but chemical relevance
> (does the reference library even contain natural-product-like structures at all?) matters more — and
> `enveda-180` fails on the second count entirely for this task.
>
> The libraries below trade a same-instrument advantage for actually containing the right kind of chemistry:
> `gnps` (largest public natural-product collection), `riken` (plant specialized-metabolite focus),
> `pluskal_ms2` (commercial/bioactive libraries), plus the smaller curated/general libraries
> (`massbank`, `mona`, `spectraverse`, `msdial`, `masaryk`) and the tiny but on-distribution
> `enveda-np-examples`. `drug_plus` (pharmaceutical drugs) is excluded for the same reason as `enveda-180` —
> wrong chemical space for a natural-products task.

First, a **metadata-only** pass over the *entire* train set (just 4 lightweight columns) for the EDA below —
this is fast because pyarrow skips reading the heavy `ms2_mzs` / `ms2_normalized_intensities` array columns.


In [ ]:
train_schema = pq.ParquetFile(TRAIN_PATH).schema.names
print(f"train.parquet has {len(train_schema)} columns:")
print(train_schema)


In [ ]:
# Lightweight full-train pass (no spectra arrays) purely for EDA / library-size sanity checks.
train_meta_cols = ["ingest_lib", "inchikey14", "adduct", "precursor_mz"]
train_meta = pd.read_parquet(TRAIN_PATH, columns=train_meta_cols)
print("train_meta shape:", train_meta.shape)

lib_stats = train_meta.groupby("ingest_lib").agg(
    n_spectra=("inchikey14", "size"),
    n_unique_structures=("inchikey14", "nunique"),
).sort_values("n_spectra", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
lib_stats["n_spectra"].plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Spectra count per ingest_lib (full train)")
axes[0].invert_yaxis()

lib_stats["n_unique_structures"].plot(kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("Unique structures (InChIKey14) per ingest_lib")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

lib_stats


In [ ]:
# Reference libraries used for the actual spectral search below, chosen for CHEMICAL relevance to the
# natural-product-focused test set (reference notes §1 "Focus"), not for instrument match.
#
# Deliberately EXCLUDED:
#   - "enveda-180": same instrument (timsTOF) as test, but synthetic drug-like screening chemistry.
#     Using it as a reference previously produced Public MRR@25 = 0.000 -- it's 94% of the combined
#     library by spectra count, so it dominated both the ranked candidates and the fallback list with
#     confidently-wrong drug-like structures while containing ~zero correct natural-product answers.
#   - "drug_plus": pharmaceutical drugs -- same problem, wrong chemical space for this task.
#
# Included libraries span different instruments (Orbitrap, QTOF, etc.) rather than test's timsTOF, which can
# slightly hurt raw cosine similarity (different resolution / typical fragmentation efficiency), but that's a
# much smaller effect than simply not containing the right chemistry at all. Extend/trim this list and re-run
# the local MRR@25 validation below to see the effect of any change.
REFERENCE_LIBS = [
    "enveda-np-examples",  # tiny (250 structures) but exact same instrument + pipeline as test
    "gnps",                # largest public natural-product collection, most heterogeneous
    "riken",                # strong plant specialized-metabolite focus
    "pluskal_ms2",          # MSnLib -- commercial/bioactive libraries, multi-CE
    "massbank",             # curated consortium library, many labs/instruments
    "mona",                 # MassBank of North America
    "spectraverse",         # recent harmonised aggregation, rarer public libraries
    "msdial",               # distributed with MS-DIAL software, multi-instrument
    "masaryk",              # small chemical-standards library
]

REF_COLUMNS = [
    "normalized_smiles", "inchikey14", "adduct", "precursor_mz",
    "ms2_mzs", "ms2_normalized_intensities", "ingest_lib", "collision_energy_ev",
]

# `filters` pushes the ingest_lib predicate down into the parquet reader (row-group + row level),
# so we never materialize the full 3GB file in memory.
ref_lib = pd.read_parquet(TRAIN_PATH, columns=REF_COLUMNS, filters=[("ingest_lib", "in", REFERENCE_LIBS)])
print("Reference library rows:", len(ref_lib))
print(ref_lib["ingest_lib"].value_counts())

ref_lib["neutral_mass"] = [
    neutral_mass_from_precursor(mz, add) for mz, add in zip(ref_lib["precursor_mz"], ref_lib["adduct"])
]
ref_lib = ref_lib.dropna(subset=["neutral_mass", "ms2_mzs", "ms2_normalized_intensities"]).reset_index(drop=True)

# Sorting by neutral_mass turns the mass-window lookup into a binary search (np.searchsorted) below.
ref_lib = ref_lib.sort_values("neutral_mass").reset_index(drop=True)
ref_masses = ref_lib["neutral_mass"].to_numpy()
print("Usable reference spectra after cleaning:", len(ref_lib))
print("Unique reference structures (InChIKey14):", ref_lib["inchikey14"].nunique())


### Proxy "findability" check (stand-in for hidden novelty classes)

The true novelty class (1/2/3, see reference notes §1) is hidden during the competition. As a cheap proxy, we
can at least check **how many test molecules have *any* reference spectrum within their mass window** — a
necessary (not sufficient) condition for a spectral-match-based method like this one to work at all. Molecules
with zero mass-window hits are very likely Novelty Class 2 or 3 for this reference library, and this baseline
will fall back to the generic "most common structure" list for them.


In [ ]:
def mass_window_bounds(neutral_mass, ref_masses_sorted, ppm_tol=15, abs_tol=0.01):
    '''Binary-search the [neutral_mass - tol, neutral_mass + tol] slice of a sorted mass array.'''
    tol = max(abs_tol, neutral_mass * ppm_tol * 1e-6)
    lo = np.searchsorted(ref_masses_sorted, neutral_mass - tol, side="left")
    hi = np.searchsorted(ref_masses_sorted, neutral_mass + tol, side="right")
    return lo, hi

has_mass_match = []
for molecule_id, group in test_df.groupby("molecule_id"):
    found = False
    for nmass in group["neutral_mass"]:
        if np.isnan(nmass):
            continue
        lo, hi = mass_window_bounds(nmass, ref_masses)
        if hi > lo:
            found = True
            break
    has_mass_match.append(found)

has_mass_match = pd.Series(has_mass_match)
print(has_mass_match.value_counts(normalize=True).rename("share_of_molecules"))

plt.figure(figsize=(4, 4))
has_mass_match.value_counts().plot(kind="bar", color=["seagreen", "indianred"], rot=0)
plt.xticks([0, 1], ["has mass-window match", "no match"])
plt.title("Test molecules with >=1 reference spectrum\nwithin the precursor-mass window")
plt.ylabel("# molecules")
plt.show()


## 5. Spectral similarity + mass-blocked candidate search

- **Blocking**: for each test spectrum, only reference spectra whose neutral mass falls within a small ppm/Da
  window are ever compared (`mass_window_bounds` above). This is what makes an otherwise O(test × 1M+) search
  run in minutes.
- **Scoring**: within a mass window, spectra are scored with a **greedy-matched cosine similarity** — peaks are
  paired from the highest-intensity product downward, each peak used at most once, then combined into a
  standard cosine score. Intensities are square-root transformed first so a single dominant base peak doesn't
  drown out everything else (a common trick in spectral library search).
- **Aggregation**: a molecule can have up to 16 spectra (different collision energies / adducts). For each
  reference structure (InChIKey14), we keep the **best** score seen across *any* (test spectrum, reference
  spectrum) pair, then rank structures by that best score.


In [ ]:
def spectral_cosine(mzs_a, ints_a, mzs_b, ints_b, tol_da=0.02):
    '''Greedy-matched cosine similarity between two centroided MS/MS spectra.

    Peaks are matched from the highest-intensity-product pair downward; each peak can only be used in one
    match (standard greedy variant of the "cosine"/"dot product" score used in spectral library search).
    Intensities are sqrt-transformed to reduce the influence of a single dominant base peak.
    '''
    mzs_a = np.asarray(mzs_a, dtype=float)
    ints_a = np.asarray(ints_a, dtype=float)
    mzs_b = np.asarray(mzs_b, dtype=float)
    ints_b = np.asarray(ints_b, dtype=float)
    if mzs_a.size == 0 or mzs_b.size == 0:
        return 0.0

    wa = np.sqrt(np.clip(ints_a, 0, None))
    wb = np.sqrt(np.clip(ints_b, 0, None))

    order_a = np.argsort(mzs_a)
    order_b = np.argsort(mzs_b)
    mzs_a, wa = mzs_a[order_a], wa[order_a]
    mzs_b, wb = mzs_b[order_b], wb[order_b]

    candidate_pairs = []
    for i in range(mzs_a.size):
        j0 = np.searchsorted(mzs_b, mzs_a[i] - tol_da, side="left")
        j = j0
        while j < mzs_b.size and mzs_b[j] <= mzs_a[i] + tol_da:
            candidate_pairs.append((wa[i] * wb[j], i, j))
            j += 1
    if not candidate_pairs:
        return 0.0

    candidate_pairs.sort(key=lambda t: t[0], reverse=True)
    used_a, used_b = set(), set()
    matched_score = 0.0
    for prod, i, j in candidate_pairs:
        if i in used_a or j in used_b:
            continue
        used_a.add(i)
        used_b.add(j)
        matched_score += prod

    denom = np.sqrt(np.sum(wa ** 2) * np.sum(wb ** 2))
    return float(matched_score / denom) if denom > 0 else 0.0


In [ ]:
# Safety cap: in dense mass regions a window could in principle contain many thousands of reference
# spectra. If that happens, keep only the MAX_CANDIDATES_PER_SPECTRUM closest-by-mass ones, to bound runtime.
MAX_CANDIDATES_PER_SPECTRUM = 4000
PPM_TOL = 15
ABS_TOL_DA = 0.01
# Reference libraries now span several instrument families (Orbitrap, QTOF, ...) in addition to test's
# timsTOF, so peak-matching tolerance is a bit looser than a same-instrument-only search would need.
# Tighten back toward 0.01-0.015 if you restrict REFERENCE_LIBS to timsTOF-only sources.
PEAK_MATCH_TOL_DA = 0.025
TOP_K = 25


def rank_candidates_for_molecule(spectra_rows, reference_df, reference_masses_sorted, top_k=TOP_K):
    '''Rank reference structures for one molecule (a list of its test spectra).

    spectra_rows: list of dicts with keys mzs, intensities, adduct, precursor_mz for ONE molecule_id.
    reference_df / reference_masses_sorted: reference library, pre-sorted by neutral_mass, and the same
        sort order's neutral_mass numpy array (for binary search).

    Returns a list of (inchikey14, best_score, representative_smiles) sorted by score descending,
    length <= top_k.
    '''
    best_score = {}
    best_smiles = {}

    for row in spectra_rows:
        nmass = neutral_mass_from_precursor(row["precursor_mz"], row["adduct"])
        if np.isnan(nmass):
            continue
        lo, hi = mass_window_bounds(nmass, reference_masses_sorted, ppm_tol=PPM_TOL, abs_tol=ABS_TOL_DA)
        if hi <= lo:
            continue

        idx = np.arange(lo, hi)
        if idx.size > MAX_CANDIDATES_PER_SPECTRUM:
            # Keep the closest-by-mass candidates if the window is unusually crowded.
            closeness = np.abs(reference_masses_sorted[idx] - nmass)
            keep = np.argsort(closeness)[:MAX_CANDIDATES_PER_SPECTRUM]
            idx = idx[keep]

        candidate_slice = reference_df.iloc[idx]
        for ref_row in candidate_slice.itertuples():
            score = spectral_cosine(
                row["mzs"], row["intensities"],
                ref_row.ms2_mzs, ref_row.ms2_normalized_intensities,
                tol_da=PEAK_MATCH_TOL_DA,
            )
            key = ref_row.inchikey14
            if key not in best_score or score > best_score[key]:
                best_score[key] = score
                best_smiles[key] = ref_row.normalized_smiles

    ranked = sorted(best_score.items(), key=lambda kv: kv[1], reverse=True)
    return [(key, score, best_smiles[key]) for key, score in ranked[:top_k]]


## 6. Global fallback candidates

For molecules with few or zero mass-window hits (likely Novelty Class 2/3 against this reference library), we
pad the ranked list up to 25 guesses with the **most frequently observed structures** in the reference library.
This is a weak signal (no molecule-specific evidence at all) but costs nothing and occasionally helps, since
the reference library is now natural-product-biased rather than dominated by unrelated drug-like screening
compounds.


In [ ]:
fallback_counts = ref_lib["inchikey14"].value_counts()
fallback_smiles_map = ref_lib.drop_duplicates("inchikey14").set_index("inchikey14")["normalized_smiles"]
fallback_candidates = [(key, fallback_smiles_map[key]) for key in fallback_counts.index[:TOP_K]]

print(f"Prepared {len(fallback_candidates)} global fallback candidates.")
print("Most common reference structures (InChIKey14 -> spectra count):")
fallback_counts.head(5)


## 7. Local validation — MRR@25 proxy (before touching the real test set)

The real test labels are of course hidden, so we validate on a **held-out slice of the reference library
itself**: randomly pick a set of reference structures (InChIKey14 groups), remove *all* of their spectra from
the reference pool, then run the exact same search-and-rank pipeline on those held-out spectra as if they were
test spectra. This mirrors the official scoring:

1. Predicted and true SMILES are both passed through RDKit **tautomer canonicalization**
   (`rdMolStandardize.TautomerEnumerator`), pinned conceptually to RDKit `2026.03.3` (see the version check
   above for the caveat if your Kaggle image ships a different version).
2. Reduced to **InChIKey14** and compared.
3. Aggregated into **MRR@25** exactly as defined in the reference notes.

Note this is a **conservative lower bound**: holding out a molecule's spectra entirely from the reference pool
simulates the hardest case (no public reference spectra available at all, i.e. Novelty Class 2/3-like). The
real test set also contains easier Novelty Class 1 molecules where a reference spectrum genuinely exists, on
which this method should score much better than what's measured here.

*This entire section requires RDKit and is automatically skipped (with a message) if RDKit isn't available in
this environment — it has no effect on the submission-building cells further below, which don't use RDKit.*


In [ ]:
if RDKIT_AVAILABLE:
    _tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

    def canonical_inchikey14(smiles):
        '''Tautomer-canonicalize a SMILES and return the first block (14 chars) of its InChIKey.
        Mirrors the competition's match criterion. Returns None for invalid/unparseable SMILES.
        '''
        if not smiles:
            return None
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            mol = _tautomer_enumerator.Canonicalize(mol)
            inchikey = Chem.MolToInchiKey(mol)
            if not inchikey or "-" not in inchikey:
                return None
            return inchikey.split("-")[0]
        except Exception:
            return None
else:
    print("RDKit unavailable -> skipping canonical_inchikey14 definition and the local MRR@25 validation below.")


In [ ]:
if RDKIT_AVAILABLE:
    N_HOLDOUT_MOLECULES = 80  # keep small: each one needs its own mass-window search against the pseudo-reference set

    unique_ref_structures = ref_lib["inchikey14"].unique()
    holdout_keys = set(np.random.choice(
        unique_ref_structures, size=min(N_HOLDOUT_MOLECULES, len(unique_ref_structures)), replace=False
    ))

    pseudo_test = ref_lib[ref_lib["inchikey14"].isin(holdout_keys)]
    pseudo_ref = ref_lib[~ref_lib["inchikey14"].isin(holdout_keys)].reset_index(drop=True)
    # pseudo_ref is a boolean-mask subset of a mass-sorted frame, so it is still sorted by neutral_mass.
    pseudo_ref_masses = pseudo_ref["neutral_mass"].to_numpy()

    print(f"Held out {len(holdout_keys)} structures / {len(pseudo_test)} spectra as pseudo-test.")
    print(f"Pseudo-reference pool: {len(pseudo_ref)} spectra.")
else:
    print("Skipped (RDKit unavailable).")


In [ ]:
if RDKIT_AVAILABLE:
    reciprocal_ranks = []

    for true_key, group in tqdm(pseudo_test.groupby("inchikey14"), desc="Local MRR@25 validation"):
        true_smiles = group["normalized_smiles"].iloc[0]
        canonical_true_key = canonical_inchikey14(true_smiles)
        if canonical_true_key is None:
            continue

        spectra_rows = [
            {"mzs": r.ms2_mzs, "intensities": r.ms2_normalized_intensities,
             "adduct": r.adduct, "precursor_mz": r.precursor_mz}
            for r in group.itertuples()
        ]
        ranked = rank_candidates_for_molecule(spectra_rows, pseudo_ref, pseudo_ref_masses)

        rank_found = None
        for i, (_, _, cand_smiles) in enumerate(ranked, start=1):
            if canonical_inchikey14(cand_smiles) == canonical_true_key:
                rank_found = i
                break
        reciprocal_ranks.append(1.0 / rank_found if rank_found else 0.0)

    mrr_at_25 = float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0
    n_scored = len(reciprocal_ranks)
    n_hit = sum(r > 0 for r in reciprocal_ranks)
    print(f"Local proxy MRR@25 (leave-one-molecule-out, n={n_scored}): {mrr_at_25:.4f}")
    print(f"Top-25 hit rate: {n_hit}/{n_scored} ({n_hit / n_scored:.1%})" if n_scored else "No molecules scored.")
else:
    print("Skipped (RDKit unavailable). Install RDKit (see the import cell above) to run this validation.")


## 8. Build the real submission

Now run the identical pipeline on the actual `test.parquet`, using the *full* reference library (no holdout
this time), and write `submission.csv`.


In [ ]:
test_records = {}
for molecule_id, group in test_df.groupby("molecule_id"):
    test_records[molecule_id] = [
        {"mzs": r.ms2_mzs, "intensities": r.ms2_normalized_intensities,
         "adduct": r.adduct, "precursor_mz": r.precursor_mz}
        for r in group.itertuples()
    ]

print(f"Prepared {len(test_records)} test molecules for ranking.")


In [ ]:
submission_rows = []

for molecule_id, spectra_rows in tqdm(test_records.items(), desc="Ranking test molecules"):
    ranked = rank_candidates_for_molecule(spectra_rows, ref_lib, ref_masses)

    seen_keys = {key for key, _, _ in ranked}
    smiles_list = [smi for _, _, smi in ranked]

    if len(smiles_list) < TOP_K:
        for key, smi in fallback_candidates:
            if key in seen_keys:
                continue
            smiles_list.append(smi)
            seen_keys.add(key)
            if len(smiles_list) >= TOP_K:
                break

    submission_rows.append({"molecule_id": molecule_id, "smiles": ";".join(smiles_list[:TOP_K])})

submission_df = pd.DataFrame(submission_rows)
print(submission_df.shape)
submission_df.head()


## 9. Validate `submission.csv` against the official rejection rules

The reference notes list the exact rejection conditions: missing columns, empty file, nulls in either column,
repeated `molecule_id`, or more than 25 semicolon-separated guesses for one molecule. This cell checks all of
them, plus full coverage of the expected `molecule_id` set, before writing the file.


In [ ]:
def validate_submission(df, expected_ids=None, max_candidates=25):
    errors = []

    if list(df.columns) != ["molecule_id", "smiles"]:
        errors.append(f"Header must be exactly ['molecule_id', 'smiles'], got {list(df.columns)}")

    if df.empty:
        errors.append("Submission is empty.")

    if df["molecule_id"].isnull().any() or (df["molecule_id"].astype(str).str.len() == 0).any():
        errors.append("Found null/empty molecule_id values.")

    if df["smiles"].isnull().any() or (df["smiles"].astype(str).str.len() == 0).any():
        errors.append("Found null/empty smiles values.")

    dup_ids = df["molecule_id"][df["molecule_id"].duplicated()]
    if len(dup_ids) > 0:
        errors.append(f"{len(dup_ids)} duplicate molecule_id rows, e.g. {dup_ids.unique()[:5].tolist()}")

    n_candidates = df["smiles"].astype(str).apply(lambda s: len(s.split(";")))
    too_many = n_candidates > max_candidates
    if too_many.any():
        errors.append(f"{int(too_many.sum())} rows exceed {max_candidates} semicolon-separated candidates.")

    if expected_ids is not None:
        got_ids = set(df["molecule_id"])
        missing = expected_ids - got_ids
        extra = got_ids - expected_ids
        if missing:
            errors.append(f"Missing {len(missing)} expected molecule_ids, e.g. {list(missing)[:5]}")
        if extra:
            errors.append(f"{len(extra)} unexpected molecule_ids not in test set, e.g. {list(extra)[:5]}")

    if errors:
        print("VALIDATION FAILED:")
        for e in errors:
            print(" -", e)
    else:
        print(f"VALIDATION PASSED: {len(df)} rows, correct header, no nulls, no duplicate molecule_id, "
              f"<= {max_candidates} candidates per row, full coverage of expected molecule_ids.")
    return len(errors) == 0


expected_ids = set(test_df["molecule_id"].unique())
is_valid = validate_submission(submission_df, expected_ids=expected_ids)
assert is_valid, "Fix the issues above before writing submission.csv"


In [ ]:
submission_df.to_csv("submission.csv", index=False)
print("Wrote submission.csv")

# Re-read from disk and re-validate, exactly the way the grader will read it.
reloaded = pd.read_csv("submission.csv", dtype=str, keep_default_na=False)
assert validate_submission(reloaded, expected_ids=expected_ids)
reloaded.head()


## 10. Ideas to improve on this baseline

- **Add `enveda-180` back in cautiously, not as the primary library**: it's still same-instrument, and a
  handful of its structures could coincidentally be right. If you want to include it, weight/rank it below the
  natural-product libraries (e.g. only use it to fill remaining fallback slots) rather than merging it into
  the main search on equal footing — mixed in at full strength it previously produced Public MRR@25 = 0.000.
- **Modified cosine**: extend `spectral_cosine` to also match peaks after shifting by the precursor mass
  difference (not just direct m/z alignment) — recovers cross-instrument / neutral-loss similarity that a
  naive cosine misses, likely the single biggest lever for Novelty Class 1 recall now that the library
  chemistry itself is fixed.
- **Formula-first filtering**: use `MIST-CF` / `SIRIUS`-style molecular-formula prediction from the spectrum to
  narrow candidates *before* (or instead of) mass-only blocking — tighter than a ppm mass window alone.
- **Self-supervised spectrum embeddings**: `GeMS` / `DreaMS` pretrained embeddings (trained on the much larger
  unannotated `MassIVE` corpus) could replace/augment the hand-built cosine similarity, especially useful for
  Novelty Class 2 molecules where no exact reference spectrum exists but a *structurally similar* one does.
- **De novo generation for Class 3**: no library-search method (including this one) can propose a structure
  that isn't already present somewhere in the reference data — Class 3 molecules require a genuinely
  generative / fragment-assembly approach.
- **Tune `PPM_TOL` / `PEAK_MATCH_TOL_DA` / `MAX_CANDIDATES_PER_SPECTRUM`** against the local MRR@25 proxy above
  before trusting them on the real test set.
